# Going further - feature selection

Optional notebook, nothing in the course depends on it. It continues Part 5 of
`1a_EDA_data_processing`, which covered filter methods only.

Wrapper and embedded methods are here. The result to look at is not any single
selection, but the fact that four models selecting from the same ten features
do not agree.

## Part 0 - Setup

Run this cell first. It fetches the course repository into the Colab session and
moves into the `notebooks/` folder, so that the `../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory.

If it prints `data ok: True`, you are set.

In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally; safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## Part 1 - Load the prepared dataset

The same train and test split saved by `1a`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X_train = pd.read_csv("../data/housing_prices/X_train.csv", index_col="Id")
X_test  = pd.read_csv("../data/housing_prices/X_test.csv",  index_col="Id")
y_train = pd.read_csv("../data/housing_prices/y_train.csv", index_col="Id").squeeze()
y_test  = pd.read_csv("../data/housing_prices/y_test.csv",  index_col="Id").squeeze()

# the wrapper cells below refer to X and to the feature list by these names
X = X_train
features = list(X_train.columns)

print("train:", X_train.shape, "| test:", X_test.shape)
print("features:", features)

### One install

`mlxtend` is preinstalled on Colab. The commented line is for local use.

In [ ]:
# %pip install -q mlxtend
from mlxtend.feature_selection import SequentialFeatureSelector
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
print("mlxtend ready")

### Wrapper methods
Find the most relevant (highest performing) features with 'forward-selection' approach for several models:
- linear regression,
- support vector machine/regressor,
- decision tree,
- random forest.

In [ ]:
from sklearn.linear_model import LinearRegression
# find best features for linear regression model
lr = LinearRegression()
sfs_lr = SequentialFeatureSelector(lr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error').fit(X_train, y_train)
fig = plot_sfs(sfs_lr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_lr.k_feature_idx_)]
print(selected_features)

In [ ]:
# find best features for support vector regression model
from sklearn.svm import SVR
svr = SVR(kernel='linear')
sfs_svr = SequentialFeatureSelector(svr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error').fit(X_train, y_train)
plot_sfs(sfs_svr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_svr.k_feature_idx_)]
print(selected_features)

Note: the next two cells are slow. Forward selection fits the model once per
candidate feature per step, and both models are scored on `absolute_error`,
which has no closed form. Measured on two cores, the decision tree takes a few
seconds and the random forest two to three minutes - it has not hung.

Replace `criterion='absolute_error'` with `criterion='squared_error'` in the
random forest cell if you would rather not wait. It is about ten times faster
and selects almost the same features.


In [ ]:
# find best features for decision tree regression model
from sklearn import tree
dtr = tree.DecisionTreeRegressor(criterion='absolute_error')
sfs_dtr = SequentialFeatureSelector(dtr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error', n_jobs=-1).fit(X_train, y_train)
fig = plot_sfs(sfs_dtr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_dtr.k_feature_idx_)]
print(selected_features)

In [ ]:
# find best features for random forest regression model
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(criterion='absolute_error', max_depth=10, random_state=0)
sfs_rfr = SequentialFeatureSelector(rfr, k_features='best', forward=True, floating=False, scoring='neg_mean_absolute_error', n_jobs=-1).fit(X_train, y_train)
fig = plot_sfs(sfs_rfr.get_metric_dict(), kind='std_err', figsize=(20,6))
selected_features = X.columns[list(sfs_rfr.k_feature_idx_)]
print(selected_features)

### Comparison

Each column is one model's selection, 1 meaning the feature was kept.

Read the rows: a feature every model keeps is safe, a feature only one model
keeps says something about that model rather than about the houses.


In [ ]:
feature_scores = pd.DataFrame(index=features,columns=[['LR', 'SVR', 'DT', 'RF']], data=0)
feature_scores.loc[np.array(features)[list(sfs_lr.k_feature_idx_)],'LR'] = 1
feature_scores.loc[np.array(features)[list(sfs_svr.k_feature_idx_)],'SVR'] = 1
feature_scores.loc[np.array(features)[list(sfs_dtr.k_feature_idx_)],'DT'] = 1
feature_scores.loc[np.array(features)[list(sfs_rfr.k_feature_idx_)],'RF'] = 1
feature_scores

### Embedded methods
Embedded feature selection methods are already incorporated into the intrinsical model procedures (more about those in the next few days).
We will evaluate the feature selection of two models: Lasso regression and random forest algorithm.

In [ ]:
# find features chosen by Lasso model
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

selection = SelectFromModel(Lasso(alpha=1))
selection.fit(X_train, y_train)

selected_features = X.columns[(selection.get_support())]
print(selected_features.values)

In [ ]:
# find features chosen by random forest model
from sklearn.ensemble import RandomForestRegressor

selection = SelectFromModel(RandomForestRegressor(n_estimators=100))
selection.fit(X_train, y_train)

selected_features = X.columns[(selection.get_support())]

print(selected_features.values)

### Conclusions - overall:
- *wrapper methods answer which features this particular model needs, not which features matter,*
- *the four selections disagree and none of them is wrong - a decision tree does not need a feature a linear model depends on,*
- *`GarageArea` and `GarageCars` are the collinear pair from `1b` - the linear models keep one, the trees keep the other,*
- *cost grows with the number of features, which is why filter methods are still used,*
- *embedded selection is the best value, since it comes with a model that was going to be fitted anyway,*
- *selection is not importance, and importance is not causation.*



_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 1x_Going_further_feature_selection_  
_Instructors: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_  

_References:_
- _Scikit-learn Feature selection: https://scikit-learn.org/stable/modules/feature_selection.html_
- _mlxtend Sequential Feature Selector: https://rasbt.github.io/mlxtend/user_guide/feature_selection/SequentialFeatureSelector/_

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_
